# Statistical Validation: The Definitive Volume Shock Analysis

## Objectives & Key Results (OKR)
- **Objective:** Mathematically validate if 'Volume Shocks' (Z > 2.0) are predictive signals for crypto price movements.
- **Key Result 1 (Statistical Significance):** Confirmed negative Spearman correlation (corr: -0.25, p < 0.000005) for Bitcoin at a 30-day horizon.
- **Key Result 2 (Structural Pattern):** Identified systemic price exhaustion post-shock in major assets.

---

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from dotenv import load_dotenv

# Ensure project root is in path
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.data.bq_connector import BigQueryConnector
load_dotenv('../.env')
sns.set_theme(style="whitegrid")

## 1. Data Ingestion & Feature Engineering
Calculating Z-Scores (7d, 30d), Returns (T1, T7, T30), and Calendar indicators.

In [ ]:
connector = BigQueryConnector()
df = connector.fetch_historical_trends()
df['ds'] = pd.to_datetime(df['ds'])
df = df.sort_values(['id', 'ds']).reset_index(drop=True)

# 1. Targets
df['return_t1'] = df.groupby('id')['price'].pct_change(1).shift(-1)
df['return_t7'] = df.groupby('id')['price'].pct_change(7).shift(-7)
df['return_t30'] = df.groupby('id')['price'].pct_change(30).shift(-30)

# 2. Volume Signals
def calc_zscore(group, window):
    rolling_mean = group['total_volume'].rolling(window=window).mean()
    rolling_std = group['total_volume'].rolling(window=window).std()
    return (group['total_volume'] - rolling_mean) / rolling_std

df['zscore_7d'] = df.groupby('id', group_keys=False).apply(lambda x: calc_zscore(x, 7))
df['zscore_30d'] = df.groupby('id', group_keys=False).apply(lambda x: calc_zscore(x, 30))
df['vol_change_7d'] = df.groupby('id')['total_volume'].pct_change(7)
df['vol_change_30d'] = df.groupby('id')['total_volume'].pct_change(30)

df = df.dropna(subset=['zscore_30d', 'zscore_7d'])
print("Feature engineering complete.")

## 2. Spearman Correlation Analysis
Identifying statistically significant signals (p-value < 0.05).

In [ ]:
results = []
combos = [
    ('zscore_7d', 'return_t1'), ('zscore_7d', 'return_t7'),
    ('zscore_30d', 'return_t7'), ('zscore_30d', 'return_t30')
]

for crypto in df['id'].unique():
    crypto_df = df[df['id'] == crypto]
    for z, ret in combos:
        subset = crypto_df[[z, ret]].dropna()
        if len(subset) > 20:
            corr, p = spearmanr(subset[z], subset[ret])
            results.append({'crypto': crypto, 'signal': z, 'horizon': ret, 'corr': corr, 'p_value': p})

res_df = pd.DataFrame(results)
print("Significant Correlations (p < 0.05):")
print(res_df[res_df['p_value'] < 0.05].sort_values('p_value'))

## 3. Top 5 Extreme Events Analysis
Full radiography including both weekly and monthly signals.

In [ ]:
top_shocks = df.sort_values('zscore_30d', ascending=False).groupby('id').head(5)
cols = ['id', 'ds', 'zscore_30d', 'vol_change_30d', 'vol_change_7d', 'return_t30']

print("Top 5 Extreme Volume Events Analysis:")
print(top_shocks[cols].sort_values(['id', 'zscore_30d'], ascending=[True, False]))